# Tamil Handwritten Word Recognition (CRNN + CTC) on Kaggle GPU

**Project:** Tamil Handwritten Character Recognition and Unicode Conversion System  
**Branch:** `feature/crnn-tamil-word-recognition`  
**Dataset:** IIIT-INDIC-HW Tamil Word Dataset (103,518 words: 75,736 Train / 11,598 Val / 16,184 Test)  
**Architecture:** VGG-style 4-block CNN + 2-layer Bidirectional LSTM + CTC Head (73 classes: 72 Tamil chars + 1 blank)  

---
### Pipeline Steps:
1. GPU & CUDA hardware acceleration verification.
2. Locate dataset from `/kaggle/input/` and extract to high-speed NVMe storage.
3. Pull latest code from GitHub.
4. Fast smoke test (2 batches) to verify AMP mixed precision and dataloader.
5. Full training loop with Resumable Checkpointing (`latest_checkpoint.pt`, `best_model.pt`).
6. Rigorous evaluation computing Character Error Rate (CER), Word Error Rate (WER), and Exact Match.
7. Bundle all artifacts into `tamil_crnn_artifacts.zip` for one-click download.

In [ ]:
# Cell 1: Hardware & GPU Environment Verification
import os
import sys
import torch

print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU: {gpu_name} ({total_mem:.2f} GB VRAM)")
    print(f"Device Count: {torch.cuda.device_count()}")
    !nvidia-smi
else:
    print("[WARNING] GPU acceleration is NOT detected! Please navigate to Notebook Settings -> Accelerator -> GPU T4 x2 or P100.")

In [ ]:
# Cell 2: Locate Dataset and Extract to Fast Scratchpad
import glob
import zipfile
import shutil
from pathlib import Path

print("=== Searching for Tamil Word Dataset in /kaggle/input ===")
possible_zips = glob.glob('/kaggle/input/**/tamil_word_dataset.zip', recursive=True)
if not possible_zips:
    possible_zips = glob.glob('/kaggle/input/**/*.zip', recursive=True)

target_zip = None
for z_path in possible_zips:
    if 'tamil' in z_path.lower() or 'indic' in z_path.lower() or 'word' in z_path.lower():
        target_zip = z_path
        break

if not target_zip and possible_zips:
    target_zip = possible_zips[0]

data_dir = '/kaggle/working/data'
os.makedirs(data_dir, exist_ok=True)

if target_zip and os.path.exists(target_zip):
    print(f"Found dataset archive: {target_zip} ({os.path.getsize(target_zip)/(1024**2):.1f} MB)")
    print(f"Extracting root files (vocab.txt, train.txt, val.txt, test.txt) to {data_dir}...")
    with zipfile.ZipFile(target_zip, 'r') as root_z:
        # Extract text files first
        for txt in ['vocab.txt', 'train.txt', 'val.txt', 'test.txt']:
            if txt in root_z.namelist():
                root_z.extract(txt, data_dir)
                print(f"  Extracted {txt}")
        
        # Extract nested image zips directly to data_dir for maximum I/O performance
        for split in ['val', 'test', 'train']:
            split_zip_name = f"{split}.zip"
            if split_zip_name in root_z.namelist():
                print(f"  Unpacking {split_zip_name} to {data_dir}/{split}/ ...")
                split_bytes = root_z.read(split_zip_name)
                import io
                with zipfile.ZipFile(io.BytesIO(split_bytes), 'r') as sz:
                    sz.extractall(data_dir)
                print(f"  Done unpacking {split_zip_name}.")
else:
    # Check if dataset was already attached in uncompressed form
    txt_candidates = glob.glob('/kaggle/input/**/vocab.txt', recursive=True)
    if txt_candidates:
        input_folder = os.path.dirname(txt_candidates[0])
        print(f"Dataset already uncompressed at: {input_folder}")
        data_dir = input_folder
    else:
        raise FileNotFoundError("Could not locate tamil_word_dataset.zip or vocab.txt in /kaggle/input!")

print(f"Dataset ready at: {data_dir}")
!df -h /kaggle/working

In [ ]:
# Cell 3: Clone / Setup Git Repository
%cd /kaggle/working
repo_dir = '/kaggle/working/tamil-handwritten-unicode'

if not os.path.exists(repo_dir):
    print("Cloning repository...")
    !git clone https://github.com/kamalakannan675/tamil-handwritten-unicode.git
    %cd {repo_dir}
    !git checkout feature/crnn-tamil-word-recognition || git checkout -b feature/crnn-tamil-word-recognition
else:
    print("Repository already exists, pulling updates...")
    %cd {repo_dir}
    !git pull

if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)

print("Repository configured successfully.")

In [ ]:
# Cell 4: Unit Test & Verification Suite
%cd {repo_dir}
print("=== Running CRNN Unit Tests ===")
!python -m unittest tests/test_crnn_dataset.py

In [ ]:
# Cell 5: Fast Smoke Test (2 Batches on GPU)
%cd {repo_dir}
print("=== Running Fast Smoke Test ===")
!python src/training/train_crnn.py \
    --extracted_dir {data_dir} \
    --vocab_path reports/tamil_ctc_char_vocab.json \
    --checkpoint_dir /kaggle/working/outputs/checkpoints_smoke \
    --batch_size 16 \
    --num_epochs 1 \
    --max_train_samples 32 \
    --max_val_samples 16

In [ ]:
# Cell 6: Full Model Training (75,736 Train / 11,598 Val)
# If Kaggle session times out, re-running this cell automatically resumes from latest_checkpoint.pt
%cd {repo_dir}
ckpt_dir = '/kaggle/working/outputs/checkpoints'
os.makedirs(ckpt_dir, exist_ok=True)

!python src/training/train_crnn.py \
    --extracted_dir {data_dir} \
    --vocab_path reports/tamil_ctc_char_vocab.json \
    --checkpoint_dir {ckpt_dir} \
    --batch_size 64 \
    --num_epochs 25 \
    --learning_rate 5e-4 \
    --hidden_size 256 \
    --patience 5 \
    --num_workers 2 \
    --resume

In [ ]:
# Cell 7: Full Validation and Test Partition Evaluation
%cd {repo_dir}
best_model_pt = f"{ckpt_dir}/best_model.pt"

print("=== 1. Evaluating on Validation Partition (11,598 samples) ===")
!python src/inference/evaluate_crnn.py \
    --checkpoint {best_model_pt} \
    --split val \
    --extracted_dir {data_dir} \
    --vocab_path reports/tamil_ctc_char_vocab.json \
    --batch_size 64 \
    --output_json /kaggle/working/outputs/evaluation_crnn_val.json

print("\n=== 2. Evaluating on Official Test Partition (16,184 samples) ===")
!python src/inference/evaluate_crnn.py \
    --checkpoint {best_model_pt} \
    --split test \
    --extracted_dir {data_dir} \
    --vocab_path reports/tamil_ctc_char_vocab.json \
    --batch_size 64 \
    --output_json /kaggle/working/outputs/evaluation_crnn_test.json

In [ ]:
# Cell 8: Bundle and Package Artifacts for Download
import zipfile
import hashlib

artifacts_zip = '/kaggle/working/tamil_crnn_artifacts.zip'
files_to_bundle = [
    f"{ckpt_dir}/best_model.pt",
    f"{ckpt_dir}/latest_checkpoint.pt",
    f"{ckpt_dir}/training_log.json",
    '/kaggle/working/outputs/evaluation_crnn_val.json',
    '/kaggle/working/outputs/evaluation_crnn_test.json',
    f"{repo_dir}/reports/tamil_ctc_char_vocab.json"
]

print(f"Bundling training artifacts into {artifacts_zip}...")
with zipfile.ZipFile(artifacts_zip, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for fpath in files_to_bundle:
        if os.path.exists(fpath):
            arcname = os.path.basename(fpath)
            zf.write(fpath, arcname=arcname)
            print(f"  Added: {arcname} ({os.path.getsize(fpath)/(1024**2):.2f} MB)")
        else:
            print(f"  [Notice] File not found (skipped): {fpath}")

# Calculate SHA-256 Checksum
sha256 = hashlib.sha256()
with open(artifacts_zip, 'rb') as f:
    for chunk in iter(lambda: f.read(65536), b''):
        sha256.update(chunk)
checksum = sha256.hexdigest()

print("\n======================================================")
print(f"Artifact Bundle Ready: {artifacts_zip}")
print(f"Size: {os.path.getsize(artifacts_zip)/(1024**2):.2f} MB")
print(f"SHA-256 Checksum: {checksum}")
print("Download this file from the Kaggle 'Output' tab on the right sidebar.")
print("======================================================")